In [1]:
import numpy as np
import pandas as pd
import re
import time
from calendar import monthrange
from datetime import datetime
from tqdm import tqdm

import requests
from bs4 import BeautifulSoup

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

<h1>Data Collection</h1>

In [2]:
with open("guardian_api_key.txt", "r") as f:
    GUARDIAN_API_KEY = f.read()

GUARDIAN_BASE_URL = "https://content.guardianapis.com/search" # we use the content endpoint to get each article body in full

with open("nyt_api_key.txt", "r") as f:
    NYT_API_KEY = f.read()
    
NYT_BASE_URL = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

In [3]:
CLIMATE_ACTIVIST_GROUPS = {
    '"Climate Activists"',
    '"Climate Protest"',
    '"Climate Strike"',
    '"Extinction Rebellion"',
    '"Just Stop Oil"',
    '"Fridays for Future"',
    '"Last Generation"',
    '"Letzte Generation"',
    '"Sunrise Movement"',
    '"350.org"',
    '"Climate Defiance"',
    '"Insulate Britain"',
    '"Ende Gelände"',
    '"Ocean Rebellion"',
    '"Scientist Rebellion"'
    } 
# a in no way exhaustive, but pretty extensive list of climate activist groups relevant for both papers

# search terms covering broad climate change, policy, energy transition terms (we add extra quotation signs for exact matches in the API, otherwise it would search for terms individually harming precision)
CLIMATE_TERMS = {
    '"climate change"', '"global warming"', "greenhouse", '"climate policy"',
    '"green energy"', "emissions", "co2", "renewable", '"electric vehicle"'
    #'"fossil fuel"', #"heatwave", "wildfire", "drought", "flooding", # these could be interesting since they can occur without a mention of climate change, but were problematic e. g. when flood was used in a different context
    #'"solar power"', '"wind power"'
}

<h2>The Guardian</h2>

Find documentation here: https://open-platform.theguardian.com/documentation

In [4]:
guardian_params = {
    #"q": " OR ".join(CLIMATE_TERMS),
    #"section": "environment",
    "from-date": "2026-01-01",
    #"to-date": "2010-12-31",
    "page-size": 200, # max page size is 200 -> maximize articles per request
    "show-fields": "body,headline,trailText",
    "api-key": GUARDIAN_API_KEY,
    "type": "article", # we don't want e. g. liveblogs which contain a huge array of different news typically -> not very relevant
    "format": "json"
}

def transform_response_to_df_guardian(response):
    """
    Transform a Guardian API JSON response into a pandas DataFrame

    Parameters:
        response (dict): 
            JSON response from the Guardian API

    Returns:
        pd.DataFrame: DataFrame with columns:
            - id
            - section_id
            - section_name
            - date
            - title
            - url
            - abstract
            - body
    """
    guardian_articles = []

    for article in response["response"]["results"]:
        guardian_articles.append({
            "id": article["id"],
            "section_id": article["sectionId"],
            "section_name": article["sectionName"],
            "date": datetime.fromisoformat(article["webPublicationDate"].replace("Z", "+00:00")), # convert data type from string to datetime
            "title": article["webTitle"],
            "url": article["webUrl"],
            # body and abstract text is in html per default, remove using BS html parser + we need to remove the "\xa0" which serves as a non-breaking space in the original text (as well as escape character "\n")
            "abstract": BeautifulSoup(article["fields"]["trailText"], "html.parser").get_text().replace("\xa0", " ").replace("\n", " "),
            "body": BeautifulSoup(article["fields"]["body"], "html.parser").get_text().replace("\xa0", " ").replace("\n", " ")
        })

    return pd.DataFrame(guardian_articles)


def get_yearly_articles_guardian(base_url, params, query_terms, from_date, to_date=None, section=None, attempts=3):
    """
    Collects Guardian articles from its API for a given query and date range

    Parameters:
        base_url (str): 
            Guardian API endpoint
        params (dict): 
            API parameters to send with request (includes API key, page-size etc)
        query_terms (list): 
            List of search terms
        from_date (str): 
            Start date in YYYY-MM-DD format
        to_date (str, optioal): 
            End date in YYYY-MM-DD format
        section (str): 
            Section filter (e.g. 'environment')

        attempts (int):
            Number of times we try to request the API again after encountering an error. After the specified number
            of attempts, the page is skipped

    Returns:
        pandas.core.frame.DataFrame: 
            DataFrame containing all articles for the given query/date range.
    """

    # set additional parameter values
    params = params.copy() # make sure we don't edit the original params
    params["q"] = " OR ".join(query_terms)
    params["from-date"] = from_date
    if to_date:
        params["to-date"] = to_date
    if section:
        params["section"] = section
    params["type"] = "article"
    
    result_dfs = [] # list to store all page dfs
    
    # we handle initial request of first page separately since we get the no of total pages from this
    params["page"] = 1 # ensure we start on page 1

    for attempt in range(attempts): # try x times (if we get an error)
        response = requests.get(base_url, params=params)
        if response.status_code == 200: # if successful, we append response to result_dfs and get the total no of pages
            response = response.json()
            result_dfs.append(transform_response_to_df_guardian(response))
            print(f"Finished page 1")
            print(f"Total no of pages: {response["response"]["pages"]}")
            break
        else:
            print(f"Error on page 1 (attempt {attempt+1}): {response.status_code} {response.text}")
            if attempt != attempts-1:
                time.sleep(1)
            else: # if we get an error on the final attempt for page 1, we return an empty dataframe
                print(f"Page 1 failed after {attempts} attempts, returning an empty df")
                return pd.DataFrame()

    # get the remaining pages, visualize progress with tqdm 
    for page in tqdm(range(2, response["response"]["pages"] + 1), desc="Fetching pages"):
        time.sleep(1) # max one request per second limit
        params["page"] = page

        for attempt in range(attempts): # try x times (if we get an error)
            response = requests.get(base_url, params=params)
            if response.status_code == 200: # on success we append the response to result_dfs and move to next page
                result_dfs.append(transform_response_to_df_guardian(response.json()))
                break
            print(f"Error on page {page} (attempt {attempt+1}): {response.status_code} {response.text}")
            if attempt != attempts-1:
                time.sleep(1)
            else:
                print(f"Page {page} failed after {attempts} attempts, skipping")

    return pd.concat(result_dfs, ignore_index=True) # combine all resulting dfs into one

In [ ]:
# Climate Terms

# we do 2010-2026 since this API is much more cooperative than the NYT one
for year in tqdm(range(2010, 2027), desc="Processing years"):
    print(f"Current Year: {year}")
    df = get_yearly_articles_guardian(GUARDIAN_BASE_URL, guardian_params, CLIMATE_TERMS, f"{year}-01-01", to_date=f"{year}-12-31")
    df.to_csv(f"data/01_raw/guardian/guardian_{year}_climate_terms.csv", index=False)

In [ ]:
# Climate Activists

for year in tqdm(range(2010, 2027), desc="Processing years"):
    print(f"Current Year: {year}")
    df = get_yearly_articles_guardian(GUARDIAN_BASE_URL, guardian_params, CLIMATE_ACTIVIST_GROUPS, f"{year}-01-01", to_date=f"{year}-12-31")
    df.to_csv(f"data/01_raw/guardian/{year}_climate_activists.csv", index=False)

Processing years:   0%|          | 0/17 [00:00<?, ?it/s]

Current Year: 2010
Finished page 1
Total no of pages: 1


Processing years:   6%|▌         | 1/17 [00:02<00:42,  2.68s/it]

Current Year: 2011
Finished page 1
Total no of pages: 1


Processing years:  12%|█▏        | 2/17 [00:04<00:35,  2.38s/it]

Current Year: 2012
Finished page 1
Total no of pages: 1


... (25 outputs left out)

Current Year: 2025
Finished page 1
Total no of pages: 2


Processing years:  94%|█████████▍| 16/17 [01:24<00:07,  7.02s/it]

Current Year: 2026
Finished page 1
Total no of pages: 1


Processing years: 100%|██████████| 17/17 [01:26<00:00,  5.10s/it]


<h2>NYT</h2>

Documentation: https://developer.nytimes.com/docs/articlesearch-product/1/overview

Allows a fixed 10 results per query


-> We take results from the 20 first pages of each month, if that many are available
-> 20 requests per month results in 200 results per month
-> 2400 max possible results for a year at 240 requests per year (we can do a max of two full years a day)

In [ ]:
# we iterate over months in a year
# for each year get the first 20 pages -> 200 articles per month -> 2400 articles per year
# at 240 requests per year

In [34]:
nyt_params = {
    "fq": " | ".join(CLIMATE_TERMS),
    "begin_date": "20200101",
    "end_date": "20200131",
    "api-key": NYT_API_KEY,
    "sort": "relevance"
}

def transform_response_to_df_nyt(response):
    """
    Transform a NYT API JSON response into a pandas DataFrame

    Parameters:
        response (dict): 
            JSON response from the NYT API

    Returns:
        pd.DataFrame: 
            DataFrame with columns:
            - id
            - section_name
            - subsection_name
            - date
            - title
            - url
            - keywords
            - abstract
    """
    articles = []

    for article in response["response"]["docs"]:
        keywords = []
        for keyword in article["keywords"]:
            keywords.append(keyword["value"])

        articles.append({
            "id": article["_id"],
            "section_name": article["section_name"],
            "subsection_name": article["subsection_name"],
            "date": datetime.fromisoformat(article["pub_date"].replace("Z", "+00:00")), # convert data type from string to datetime
            "title": article["headline"]["main"],
            "url": article["web_url"],
            "keywords": ", ".join(keywords),
            "abstract": article["abstract"]
        })

    return pd.DataFrame(articles)

def get_articles_nyt(base_url, params, query_terms, from_date, to_date=None, section=None, attempts=3):
    """
    Collects NYT articles from its API for a given query and date range. To reasonably adhere to rate limits, it collects just
    200 results for a single request

    Parameters:
        base_url (str): 
            NYT API endpoint

        params (dict): 
            API parameters to send with request (includes API key, sorting method)

        query_terms (list): 
            List of search terms

        from_date (str): 
            Start date in YYYYMMDD format

        to_date (str, optioal): 
            End date in YYYYMMDD format

        section (str): 
            Section filter (e.g. 'environment')

        attempts (int):
            Number of times we try to request the API again after encountering an error. After the specified number
            of attempts, the page is skipped

    Returns:
        pd.DataFrame: DataFrame with columns:
            - id
            - section_id
            - section_name
            - date
            - title
            - url
            - abstract
            - body
    """
    params = params.copy() # make sure we don't edit the original params
    params["fq"] = f'Article.body:({" OR ".join(query_terms)})' # search for query terms in article body
    params["begin_date"] = from_date
    if to_date:
        params["end_date"] = to_date

    result_dfs = [] # list to store all page dfs
    
    # we handle initial request of first page separately since we get the no of total hits from this
    params["page"] = 0 # ensure we start on page 0 (0 indexing for nyt api)

    for attempt in range(attempts): # try x times (if we get an error)
        response = requests.get(base_url, params=params)
        if response.status_code == 200:
            response = response.json()
            result_dfs.append(transform_response_to_df_nyt(response))
            total_hits = response["response"]["metadata"]["hits"]
            print(f"Total no of hits: {total_hits}")            
            break
        else:
            print(f"Error on page 1 (attempt {attempt+1}): {response.status_code} {response.text}")
            if attempt != attempts-1:
                time.sleep(12)
            else: # if we get an error on the final attempt for page 1, we return an empty dataframe
                print(f"Page 1 failed after {attempts} attempts, returning an empty df")
                return pd.DataFrame()

    # account for the case of less than 200 hits -> less pages potentially
    if total_hits < 200:
        page_range = range(1, int(np.ceil(total_hits / 10)))
    else:
        page_range = range(1, 20)

    # get the remaining pages, visualize progress with tqdm 
    for page in tqdm(page_range, desc="Getting pages"):
        time.sleep(12) # need a 12s delay between each request as to not exceed the 5 requests per minute limit
        params["page"] = page
        
        for attempt in range(attempts):  # try x times (if we get an error)
            response = requests.get(base_url, params=params)
            if response.status_code == 200:
                result_dfs.append(transform_response_to_df_nyt(response.json()))
                break
            print(f"Error on page {page} (attempt {attempt+1}): {response.status_code} {response.text}")
            if attempt != attempts:
                time.sleep(12)  
            else:
                print(f"Page {page} failed after {attempts} attempts, skipping")

    return pd.concat(result_dfs, ignore_index=True) # combine all resulting dfs into one

In [33]:
def get_articles_by_month_nyt(year, from_month, to_month, base_url, params, query_terms, section=None, attempts=3):
    """
    Collects NYT articles from its API for a given query and month range within a year. Useful to comfortably get data
    equally sampled across months within a year

    Parameters:
        year (int): 
            Year to iterate over

        from_month (int):
            Starting month to iterate from

        to_month (int):
            Last month to iterate to

        base_url (str): 
            NYT API endpoint

        params (dict): 
            API parameters to send with request (includes API key, sorting method)

        query_terms (list): 
            List of search terms

        section (str): 
            Section filter (e.g. 'environment')

        attempts (int):
            Number of times we try to request the API again after encountering an error. After the specified number
            of attempts, the page is skipped

    Returns:
        pd.DataFrame: 
            DataFrame with columns:
            - id
            - section_id
            - section_name
            - date
            - title
            - url
            - abstract
            - body
    """
    monthly_dfs = []

    for month in tqdm(range(from_month, to_month+1), desc="Processing months"):
        print(f"Current month: {month}")
        
        last_day = monthrange(year, month)[1] # used to find the month length since e. g. february can vary ofc
        from_date = f"{year}{month:02d}01" # use 0 padding and 2 digit formatting of month in date string
        to_date   = f"{year}{month:02d}{last_day}"

        month_df = get_articles_nyt(base_url, params, query_terms, from_date, to_date, section, attempts)
        monthly_dfs.append(month_df)
    
    return pd.concat(monthly_dfs)


<h3>Climate Terms</h3>

Due to the rate limitation, we use staging 

In [ ]:
df_2020 = get_articles_by_month_nyt(2020, 1, 12, NYT_BASE_URL, nyt_params, CLIMATE_TERMS)
df_2020.to_csv("data/01_raw/nyt/2020_climate_terms.csv", index=False)

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Current month: 1
Total no of hits: 309


Processing months:   8%|▊         | 1/12 [03:59<43:56, 239.67s/it]

Current month: 2
Total no of hits: 293


Error on page 2 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months:  17%|█▋        | 2/12 [08:10<41:03, 246.39s/it]

... (20 outputs left out)

Error on page 2 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months:  92%|█████████▏| 11/12 [44:20<04:07, 247.56s/it]

Current month: 12
Total no of hits: 277


Processing months: 100%|██████████| 12/12 [48:19<00:00, 241.59s/it]


In [ ]:
df_2021 = get_articles_by_month_nyt(2021, 1, 12, NYT_BASE_URL, nyt_params, CLIMATE_TERMS)
df_2021.to_csv("data/01_raw/nyt/2021_climate_terms.csv", index=False)

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Current month: 1
Total no of hits: 296


Processing months:   8%|▊         | 1/12 [03:59<43:57, 239.76s/it]

Current month: 2
Total no of hits: 324


Processing months:  17%|█▋        | 2/12 [07:58<39:50, 239.02s/it]

Current month: 3
Total no of hits: 282


... (21 outputs left out)

Processing months:  92%|█████████▏| 11/12 [45:06<04:10, 250.98s/it]

Current month: 12
Total no of hits: 293


Error on page 2 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months: 100%|██████████| 12/12 [49:18<00:00, 246.53s/it]


In [ ]:
df_2022 = get_articles_by_month_nyt(2022, 1, 12, NYT_BASE_URL, nyt_params, CLIMATE_TERMS)
df_2022.to_csv("data/01_raw/nyt/2022_climate_terms.csv", index=False)

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Current month: 1
Total no of hits: 227


Processing months:   8%|▊         | 1/12 [03:59<43:51, 239.25s/it]

Current month: 2
Total no of hits: 249


Error on page 4 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months:  17%|█▋        | 2/12 [08:11<41:09, 246.90s/it]

... (24 outputs left out)

Processing months:  92%|█████████▏| 11/12 [46:00<04:11, 251.85s/it]

Current month: 12
Total no of hits: 256


Error on page 4 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months: 100%|██████████| 12/12 [50:13<00:00, 251.10s/it]


In [ ]:
df_2023 = get_articles_by_month_nyt(2023, 1, 12, NYT_BASE_URL, nyt_params, CLIMATE_TERMS)
df_2023.to_csv("data/01_raw/nyt/2023_climate_terms.csv", index=False)

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Current month: 1
Total no of hits: 270


Processing months:   8%|▊         | 1/12 [03:58<43:46, 238.75s/it]

Current month: 2
Total no of hits: 214


Processing months:  17%|█▋        | 2/12 [07:57<39:45, 238.57s/it]

Current month: 3
Total no of hits: 272


... (23 outputs left out)

Processing months:  92%|█████████▏| 11/12 [45:37<04:10, 250.82s/it]

Current month: 12
Total no of hits: 279


Error on page 3 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months: 100%|██████████| 12/12 [49:48<00:00, 249.06s/it]


In [ ]:
df_2024 = get_articles_by_month_nyt(2024, 1, 12, NYT_BASE_URL, nyt_params, CLIMATE_TERMS)
df_2024.to_csv("data/01_raw/nyt/2024_climate_terms.csv", index=False)

Processing months:   0%|          | 0/12 [00:00<?, ?it/s]

Current month: 1
Total no of hits: 251


Error on page 2 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months:   8%|▊         | 1/12 [04:11<46:04, 251.28s/it]

Current month: 2
Total no of hits: 213


Processing months:  17%|█▋        | 2/12 [08:09<40:36, 243.67s/it]

... (23 outputs left out)

Processing months:  92%|█████████▏| 11/12 [45:38<04:10, 250.98s/it]

Current month: 12
Total no of hits: 194


Error on page 4 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Processing months: 100%|██████████| 12/12 [49:49<00:00, 249.14s/it]


In [ ]:
df_2025 = get_articles_by_month_nyt(2025, 1, 12, NYT_BASE_URL, nyt_params, CLIMATE_TERMS)
df_2025.to_csv("data/01_raw/nyt/2025_climate_terms.csv", index=False)

<h3>Climate Activist</h3>

we get a lot less total hits for climate activism so we don't have to query by month

In [ ]:
df_2018 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20180101", to_date="20181231")
df_2018.to_csv("data/01_raw/nyt/2018_climate_activists.csv", index=False)

Total no of hits: 45


Getting pages: 100%|██████████| 4/4 [00:50<00:00, 12.71s/it]


In [ ]:
# 2019 is a particularly interesting year for climate activism so we start here (maybe add 2018 too if we have time)
df_2019 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20190101", to_date="20191231")
df_2019.to_csv("data/01_raw/nyt/2019_climate_activists.csv", index=False)

Total no of hits: 177


Getting pages:  18%|█▊        | 3/17 [00:37<02:55, 12.55s/it]

Error on page 4 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Getting pages: 100%|██████████| 17/17 [03:44<00:00, 13.23s/it]


In [ ]:
df_2020 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20200101", to_date="20201231")
df_2020.to_csv("data/01_raw/nyt/2020_climate_activists.csv", index=False)

Error on page 1 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}
Total no of hits: 175


Getting pages: 100%|██████████| 17/17 [03:32<00:00, 12.52s/it]


In [ ]:
df_2021 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20210101", to_date="20211231")
df_2021.to_csv("data/01_raw/nyt/2021_climate_activists.csv", index=False)

Total no of hits: 182


Getting pages:   0%|          | 0/18 [00:00<?, ?it/s]

Error on page 1 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Getting pages: 100%|██████████| 18/18 [03:57<00:00, 13.19s/it]


In [ ]:
df_2022 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20220101", to_date="20221231")
df_2022.to_csv("data/01_raw/nyt/2022_climate_activists.csv", index=False)

Total no of hits: 185


Getting pages:   0%|          | 0/18 [00:00<?, ?it/s]

Error on page 1 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Getting pages: 100%|██████████| 18/18 [03:58<00:00, 13.26s/it]


In [ ]:
df_2023 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20230101", to_date="20231231")
df_2023.to_csv("data/01_raw/nyt/2023_climate_activists.csv", index=False)

Total no of hits: 174


Getting pages:   0%|          | 0/17 [00:00<?, ?it/s]

Error on page 1 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Getting pages: 100%|██████████| 17/17 [04:00<00:00, 14.14s/it]


In [ ]:
df_2024 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20240101", to_date="20241231")
df_2024.to_csv("data/01_raw/nyt/2024_climate_activists.csv", index=False)

Total no of hits: 145


Getting pages:   0%|          | 0/14 [00:00<?, ?it/s]

Error on page 1 (attempt 1): 429 {"fault":{"faultstring":"Rate limit quota violation. Quota limit  exceeded. Identifier : e306f1db-7f5c-4c0d-92ab-c2c5737e2cc1","detail":{"errorcode":"policies.ratelimit.QuotaViolation"}}}


Getting pages: 100%|██████████| 14/14 [03:08<00:00, 13.45s/it]


In [ ]:
df_2025 = get_articles_nyt(NYT_BASE_URL, nyt_params, CLIMATE_ACTIVIST_GROUPS, from_date="20250101", to_date="20251231")
df_2025.to_csv("data/01_raw/nyt/2025_climate_activists.csv", index=False)

Total no of hits: 82


Getting pages: 100%|██████████| 8/8 [01:41<00:00, 12.66s/it]


<h1>Pre-Processing</h1>

<h3>The Guardian</h3>

<h4>Climate Terms</h4>

In [5]:
guardian_dfs = [pd.read_csv(f"data/01_raw/guardian/{year}_climate_terms.csv") for year in range(2010, 2026)]

# check if the year range worked correctly
for df, year in zip(guardian_dfs, range(2010, 2026)):
    df["date"] = pd.to_datetime(df["date"])
    if df["date"].max().year != year or df["date"].min().year != year:
        print(f"Problem with year range in {year}")

In [29]:
full_guardian_df = pd.concat(guardian_dfs)
full_guardian_df.to_csv("data/01_raw/guardian/full_2010_2025_climate_terms.csv")

In [30]:
# transform datetime dtype
full_guardian_df["date"] = pd.to_datetime(full_guardian_df["date"])

# all other cols are strings
full_guardian_df.dtypes

id                           object
section_id                   object
section_name                 object
date            datetime64[ns, UTC]
title                        object
url                          object
abstract                     object
body                         object
dtype: object

In [31]:
# inspect df lengths
df_lengths = []
for df in guardian_dfs:
    print(len(df))
    df_lengths.append(len(df))
print(f"Mean no of articles per year: {np.mean(df_lengths)}")
print(f"Total no of articles: {len(full_guardian_df)}")

3758
3854
3626
4108
4838
5666
4808
4289
3724
5714
4527
5667
5550
5142
4664
3938
Mean no of articles per year: 4617.0625
Total no of articles: 73873


In [ ]:
# We find only a few null values in abstract and body
print(full_guardian_df.isna().sum())
full_guardian_df.fillna(" ", inplace=True) # we just fill with emtpy strings

#display(full_guardian_df[full_guardian_df["abstract"].isna()])
#display(full_guardian_df[full_guardian_df["body"].isna()])

# find no duplicates
display(full_guardian_df[full_guardian_df.duplicated(keep="first")].sort_values(by=["date","id"]))


id              0
section_id      0
section_name    0
date            0
title           0
url             0
abstract        8
body            4
dtype: int64


,id,section_id,section_name,date,title,url,abstract,body


In [ ]:
# save pre-processed df
full_guardian_df.to_csv("data/02_pre-processed/guardian_2010_2025_climate_terms.csv", index=False)

<h5>ClimateBERT</h5>

https://huggingface.co/climatebert/distilroberta-base-climate-detector

For better precision, we apply climateBERT classifier to identify whether articles are climate-related or not

(this is especially necessary for sentiment analysis as well which has a 512 token limit too.
For NYT we will only be able to assess sentiment from title and abstract)

In [ ]:
# code adapted from huggingface example
model_name = "climatebert/distilroberta-base-climate-detector"

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name, max_len=512) # max length enforced by climateBERT

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, truncation=True)

# detects whether a text is climate related or not -> useful to filter out non-climate articles

# test on a known climate article
result = pipe("Scientists warn that carbon emissions must be cut by 50% before 2030.")
print(result)

# test on a non-climate article  
result = pipe("The stock market rose sharply following the Federal Reserve announcement.")
print(result)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: climatebert/distilroberta-base-climate-detector
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'yes', 'score': 0.9968528151512146}]
[{'label': 'no', 'score': 0.9754741787910461}]


We use title, abstract, and body. Since climateBERT has a token limit of 512, we can only use the truncated initial part of the full text. However, this will allow for more accurate classifications if relevant climate change vocabulary is not used in the title or abstract. If the initial part of the full text does so, we can quite confidently say this it is relevant.

In [ ]:
for year in range(2010, 2026):

    year_df = full_guardian_df[full_guardian_df["date"].dt.year == year]

    texts = (year_df["title"] + ". " + year_df["abstract"] + ". " + year_df["body"]).tolist() 
    results = []

    for i in tqdm(range(0, len(texts), 32)): # run in batches of 32 to speed up process slightly using more memory
        batch = texts[i:i+32]
        results.extend(pipe(batch)) 

    year_df["climateBERT_classification"] = [r["label"] for r in results]
    year_df["climateBERT_score"] = [r["score"] for r in results]

    year_df.to_csv(f"data/02_pre-processed/with_climate_bert/guardian_{year}_climate_terms_with_climateBERT_class.csv", index=False)

In [116]:
dfs = []
for year in range(2010, 2026):
    year_df = pd.read_csv(f"data/02_pre-processed/with_climate_bert/guardian_{year}_climate_terms_with_climateBERT_class.csv")
    dfs.append(year_df)

full_df = pd.concat(dfs)
full_df.to_csv("data/02_pre-processed/with_climate_bert/guardian_2010_2025_climate_terms_with_climateBERT_class.csv")

In [117]:
full_df["climateBERT_classification"].value_counts() / len(full_df)

climateBERT_classification
yes    0.780084
no     0.219916
Name: count, dtype: float64

In [58]:
year_df["climateBERT_classification"].value_counts()

climateBERT_classification
yes    2944
no      995
Name: count, dtype: int64

<h4>Climate Activists</h4>

In [44]:
guardian_act_dfs = [pd.read_csv(f"data/01_raw/guardian/{year}_climate_activists.csv") for year in range(2010, 2027)]
full_guardian_act_df = pd.concat(guardian_act_dfs)
full_guardian_act_df.to_csv("data/01_raw/guardian/full_2010_2026_climate_activists.csv")

In [45]:
# check if the year range worked correctly
for df, year in zip(guardian_act_dfs, range(2010, 2027)):
    df["date"] = pd.to_datetime(df["date"])
    if df["date"].max().year != year or df["date"].min().year != year:
        print(f"Problem with year range in {year}")

In [46]:
df_lengths = []
for df in guardian_act_dfs:
    print(len(df))
    df_lengths.append(len(df))
print(f"Mean no of articles per year: {np.mean(df_lengths)}")
print(f"Total no of articles: {len(full_guardian_act_df)}")

65
55
59
90
92
205
115
80
137
944
450
522
579
612
409
278
28
Mean no of articles per year: 277.6470588235294
Total no of articles: 4720


In [ ]:
# We find no null values
print(full_guardian_act_df.isna().sum())

# no duplicates
display(full_guardian_act_df[full_guardian_act_df.duplicated(keep="first")].sort_values(by=["date","id"])) 

id              0
section_id      0
section_name    0
date            0
title           0
url             0
abstract        0
body            0
dtype: int64


,id,section_id,section_name,date,title,url,abstract,body


In [48]:
# transform datetime dtype
full_guardian_act_df["date"] = pd.to_datetime(full_guardian_act_df["date"], utc=True).dt.tz_localize(None)

# all others are strings
full_guardian_act_df.dtypes

id                      object
section_id              object
section_name            object
date            datetime64[ns]
title                   object
url                     object
abstract                object
body                    object
dtype: object

In [42]:
full_guardian_act_df.describe()

,date
count,4720
mean,2020-09-24 23:10:21.001059328
min,2010-01-12 15:30:01
25%,2019-05-19 07:45:30.750000128
50%,2021-03-25 19:47:59.500000
75%,2023-04-24 00:41:08.249999872
max,2026-03-05 07:00:53


In [49]:
# save pre-processed df
full_guardian_act_df.to_csv("data/02_pre-processed/guardian_2010_2026_climate_activists.csv", index=False)

<h3>NYT</h3>

<h4>Climate Terms</h4>

In [ ]:
nyt_dfs = [pd.read_csv(f"data/01_raw/nyt/{year}_climate_terms.csv") for year in range(2020, 2026)]
full_nyt_df = pd.concat(nyt_dfs)
full_nyt_df.to_csv("data/01_raw/nyt/full_2020_2025_climate_terms.csv", index=False)

In [51]:
full_nyt_df

,id,section_name,subsection_name,date,title,url,keywords,abstract
0,nyt://article/a2e65924-de68-5bed-b00e-4424c4f3...,Arts,NaN,2020-01-30 23:12:57+00:00,Things to Do in N.Y.C. This February,https://www.nytimes.com/2020/01/30/arts/things...,New York City,"A month’s worth of free, family-friendly and o..."
1,nyt://article/54877656-88bd-5f19-9077-de839ac2...,Business,Economy,2020-01-30 20:02:34+00:00,"California, Mired in a Housing Crisis, Rejects...",https://www.nytimes.com/2020/01/30/business/ec...,"California, Real Estate and Housing (Residenti...",A lawmaker’s push for denser development near ...
2,nyt://article/329fe101-fb79-551c-8d34-fc608a0b...,Climate,NaN,2020-01-30 14:11:27+00:00,Trump Administration Moves to Relax Rules Agai...,https://www.nytimes.com/2020/01/30/climate/tru...,"Birds, Animal Migration, Animal Abuse, Rights ...",A proposed regulation would codify a legal opi...
3,nyt://article/be382c87-a88a-5444-a61a-e63d7caf...,Business,Media,2020-01-30 13:39:50+00:00,‘Every Day Matters’: Guardian Stops Accepting ...,https://www.nytimes.com/2020/01/30/business/me...,"Newspapers, Guardian (British Newspaper), Adve...",It said the decision was based on the efforts ...
4,nyt://article/9b6fdfa3-6ee3-5ba3-bd62-ed21b0e9...,Business,DealBook,2020-01-30 12:11:30+00:00,Elon Musk Hits Back at Critics,https://www.nytimes.com/2020/01/30/business/de...,"Tesla Motors Inc, Musk, Elon",Tesla reported a $105 million quarterly profit...
...,...,...,...,...,...,...,...,...
2014,nyt://article/8cc584c5-0692-5d25-9764-bdbe5d05...,Climate,NaN,2025-11-21 16:25:14+00:00,"China Offers Panda Totes, but No New Commitmen...",https://www.nytimes.com/2025/11/21/climate/chi...,"COP30, United Nations Framework Convention on ...",The United States has retreated on climate. Ch...
2015,nyt://article/4394de19-5d82-5f62-875a-ce527721...,Business,DealBook,2025-11-21 13:01:42+00:00,Three Potential Factors Behind the Market Turmoil,https://www.nytimes.com/2025/11/21/business/de...,"Stocks and Bonds, Standard & Poor's 500-Stock ...","Growing jitters over the Federal Reserve, cryp..."
2016,nyt://article/d86b8310-98b4-5b90-aa43-3144d4af...,World,Europe,2025-11-21 10:00:22+00:00,She Led Finland Through Covid and Russian Thre...,https://www.nytimes.com/2025/11/21/world/europ...,"Marin, Sanna, Finland, Politics and Government...",Sanna Marin was celebrated as a pathbreaking f...
2017,nyt://article/314e6511-aeea-5005-91c5-3fb52976...,Climate,NaN,2025-11-20 22:38:55+00:00,A Trump Overhaul of the Energy Dept. Breaks Up...,https://www.nytimes.com/2025/11/20/climate/cle...,"Energy Department, Global Warming, Greenhouse ...",The reorganization reflects the ongoing shift ...


In [ ]:
# check if the year range worked correctly and adjust errors
current_index = 0
for df, year in zip(nyt_dfs, range(2019, 2026)):
    df["date"] = pd.to_datetime(df["date"])
    entries_outside_year = len(df[df["date"].dt.year != year])
    if entries_outside_year > 0:
        print(f"Problem with year range in {year}. There are {entries_outside_year} entries outside the year range.")
        nyt_dfs[current_index] = df[df["date"].dt.year == year]
    current_index += 1

full_nyt_df = pd.concat(nyt_dfs)
full_nyt_df.to_csv("data/01_raw/nyt/full_2020_2025_climate_terms.csv", index=False)

In [55]:
df_lengths = []
for df in nyt_dfs:
    print(len(df))
    df_lengths.append(len(df))
print(f"Mean no of articles per year: {np.mean(df_lengths)}")
print(f"Total no of articles: {len(full_nyt_df)}")

2352
2399
2399
2399
2391
2019
Mean no of articles per year: 2326.5
Total no of articles: 13959


In [56]:
# We find a lot of null values for subsections which is logical
# lots of missing keywords and abstracts which is acceptable though
print(full_nyt_df.isna().sum())
full_nyt_df.fillna(" ", inplace=True)

#display(full_guardian_df[full_guardian_df["abstract"].isna()])
#display(full_guardian_df[full_guardian_df["body"].isna()])

# produced only a single duplicate that we remove
display(full_nyt_df[full_nyt_df.duplicated(keep="first")].sort_values(by=["date","id"])) 
full_nyt_df.drop_duplicates(inplace=True)

id                    0
section_name          0
subsection_name    9380
date                  0
title                 0
url                   0
keywords           1919
abstract            713
dtype: int64


,id,section_name,subsection_name,date,title,url,keywords,abstract
1735,nyt://article/bbd2648f-5eb0-50c6-81fa-4525d5dd...,Climate,,2024-09-16 16:38:45+00:00,Join the Times’s Climate Event With Jane Gooda...,https://www.nytimes.com/2024/09/16/climate/joi...,"Global Warming, Goodall, Jane, New York Times,...",Sign up to watch the livestream of the Climate...


In [57]:
# top 10 sections
full_nyt_df["section_name"].value_counts().head(10)

section_name
U.S.                    2263
Climate                 1974
Business                1935
Opinion                 1852
World                   1553
Briefing                1244
New York                 368
Arts                     357
Books                    267
The Learning Network     263
Name: count, dtype: int64

In [58]:
# transform datetime dtype
full_nyt_df["date"] = pd.to_datetime(full_nyt_df["date"], utc=True).dt.tz_localize(None)

# all others are strings
full_nyt_df.dtypes

id                         object
section_name               object
subsection_name            object
date               datetime64[ns]
title                      object
url                        object
keywords                   object
abstract                   object
dtype: object

In [ ]:
# save pre-processed df
full_nyt_df.to_csv("data/02_re-processed/nyt_2020_2025_climate_terms.csv", index=False)

<h5>ClimateBERT</h5>

WE DO NOT HAVE THE BODY SO WE CANNOT USE THE INITIAL TEXT OF THE ARTICLE -> WE DEEM CLIMATEBERT INCAPABLE OF GUESSING WHETHER CLIMATE RELATED ENOUGH OR NOT FOR NYT

<h4>Climate Activists</h4>

In [9]:
nyt_act_dfs = [pd.read_csv(f"data/01_raw/nyt/{year}_climate_activists.csv") for year in range(2018, 2026)]

# check if the year range worked correctly
for df, year in zip(nyt_act_dfs, range(2018, 2026)):
    df["date"] = pd.to_datetime(df["date"])
    if df["date"].max().year != year or df["date"].min().year != year:
        print(f"Problem with year range in {year}")

In [11]:
full_nyt_act_df = pd.concat(nyt_act_dfs)
full_nyt_act_df.to_csv("data/01_raw/nyt/full_2018_2025_climate_terms.csv", index=False)

In [12]:
df_lengths = []
for df in nyt_act_dfs:
    print(len(df))
    df_lengths.append(len(df))
print(f"Mean no of articles per year: {np.mean(df_lengths)}")
print(f"Total no of articles: {len(full_nyt_act_df)}")

45
177
175
182
185
174
145
82
Mean no of articles per year: 145.625
Total no of articles: 1165


In [13]:
# We find some null values
print(full_nyt_act_df.isna().sum())
full_nyt_act_df.fillna(" ", inplace=True)

# no duplicates
display(full_nyt_act_df[full_nyt_act_df.duplicated(keep="first")].sort_values(by=["date","id"])) # the api search produced almost 2000 duplicates -> need to remove

id                   0
section_name         0
subsection_name    729
date                 0
title                0
url                  0
keywords           118
abstract            54
dtype: int64


,id,section_name,subsection_name,date,title,url,keywords,abstract


In [14]:
# transform datetime dtype
full_nyt_act_df["date"] = pd.to_datetime(full_nyt_act_df["date"], utc=True).dt.tz_localize(None)

# all others are strings
full_nyt_act_df.dtypes

id                         object
section_name               object
subsection_name            object
date               datetime64[ns]
title                      object
url                        object
keywords                   object
abstract                   object
dtype: object

In [15]:
# save pre-processed df
full_nyt_act_df.to_csv("data/02_pre-processed/nyt_2018_2025_climate_activists.csv", index=False)